# ラベルなしデータを用いたVision Transformerの事前学習

Vision Transformer (ViT) は大量のデータを用いて事前学習を行うことで，高い認識性能を発揮することが知られています．\
一方で，ViTの一般的な学習方法である教師あり学習は，人手により正解情報を付与したラベルありデータを必要とします．\
そのため，大量のラベルありデータを用いた教師あり学習による事前学習の実現は，多くの人的・時間的コストを必要とします．\
このような問題から，ラベルなしデータを用いた事前学習方法である自己教師あり学習を活用した手法が盛んに研究されています．

自己教師あり学習は，ラベルなしデータを用いて擬似的な問題 (Pretextタスク) によりモデルを事前学習する方法です．\
様々な自己教師あり学習の手法が提案されていますが，今回はViTのための自己教師あり学習として提案された「Masked Autoencoders (MAE)」について紹介すると共に，簡単な問題設定から実際に評価を行います．

自己教師あり学習の概要や代表的な自己教師あり学習の1つである対照学習については，以下のプログラムで取り扱っています．\
[MPRGDeepLearningLectureNotebook：14_self_supervised_learning](https://colab.research.google.com/github/machine-perception-robotics-group/MPRGDeepLearningLectureNotebook/blob/master/11_cnn_pytorch/14_self_supervised_learning.ipynb#scrollTo=HpXJ4V8D2Cc_
)

## Masked Autoencoders (MAE)
MAEは，自然言語処理分野のTransformerの事前学習方法であるMasked Language Modeling (MLM) を画像へ応用したMasked Image Modeling (MIM) による自己教師あり学習方法です．\
パッチ分割した入力画像に対して，パッチ単位でランダムにマスク処理を行い，マスクしたパッチのピクセルを予測することで学習を行います．\
そのため，MAEはマスクをしていないパッチの情報からマスクしたパッチの情報を予測するような学習となります．

### ネットワーク構造
MAEのネットワークは，学習対象のViTであるEncoderと小規模なViTであるDecoderから構成されます．\
Encoderは，マスクが適用されなかったパッチを入力として，各パッチの特徴量を抽出します．\
Decoderは，エンコーダが抽出したパッチの特徴量（水色のパッチ）とマスクしたパッチであることを表すマスクトークン（灰色のパッチ）を入力として，各パッチのピクセルを出力します．\
Encoderへマスクが適用されなかったパッチのみを入力することで，学習時間の短縮やメモリ消費量の削減を実現しています．\
自己教師あり学習による事前学習後は，Decoderを破棄してEncoderのみを目的のタスク（下流タスク）へ利用します．

<img src="https://dl.dropboxusercontent.com/s/pz9g73wwbrxuc56/MPRG_github_MAE_model.png" width = 50%>

### 学習方法
パッチに対して高い確率（MAEの論文内の実験では75%に設定）でマスク処理を適用します．\
損失式には平均二乗誤差 (MSE) を使用し，マスクを適用したパッチに対してのみ損失計算を行います．

MAEの学習の流れは以下のようになります．
1. 入力画像をパッチに分割
2. 各パッチに対してパッチ単位でランダムにマスク処理
2. Encoder・Decoderにより各パッチのピクセルを予測
3. マスクを適用したパッチに対してのみを損失を計算
4. 損失値が小さくなるようにEncoder・Decoderを学習

# MAEによるViTの事前学習
クラス分類問題において，MAEにより自己教師あり学習したViTをfine-tuningにより評価を行います．\
ViTの事前学習に使用される代表的なデータセットとして，約128万サンプルの画像から構成されたImageNet-1Kがあります．\
しかし，ImageNet-1Kは，約140GBのデータセットであり，データセットの準備（ダウンロード）とImageNet-1Kを用いた学習に必要とする時間からGoogle Colaboratory上で学習を行うことは困難です．\
そこで今回は，MAEの著者らが公開しているImageNet-1Kを用いてMAEにより学習したEncoderの重みパラメータをダウンロードして，CIFAR-10データセットへfine-tuningします．

プログラムの構成は以下の通りです．
1. MAEの学習結果の確認：学習済みのEncoder・Decoderによる再構成画像の可視化
2. ImageNetを用いて教師あり事前学習したViTのfine-tuning
3. ImageNetを用いてMAEにより事前学習したViTのfine-tuning

2と3の結果を比較することで，事前学習の方法による精度の違いを比較します．

## 必要なモジュールの読み込み

MAEに対応したネットワークは，Encoder，Decoder，マスクの適用など様々な処理の定義が必要なため，非常に長いプログラムとなります．\
そこで，今回はMAEの著者らが公開しているMAEのプログラム ([GitHub](https://github.com/facebookresearch/mae)) をダウンロードして，ネットワークが定義されているプログラム ([models_mae.py](https://github.com/facebookresearch/mae/blob/main/models_mae.py)) をimportします．

まず初めにMAEの動作に必要な環境の構築 (pip install) とMAEのプログラムのダウンロード (git clone) を行います．

In [ ]:
import sys

!pip install japanize-matplotlib
!pip3 install timm==0.4.5
!git clone https://github.com/facebookresearch/mae.git
sys.path.append('./mae')

# mae/util/pos_embed.py が古いnumpyでのみ有効なnp.float（float型の非推奨エイリアス）を使用しているため，
# 現在のnumpyでも動作するように修正しておきます．
!sed -i 's/np\.float)/float)/' mae/util/pos_embed.py

必要なモジュールを読み込みます．

In [ ]:
import requests

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

import timm
from timm.models.layers import trunc_normal_, DropPath
from timm.optim import create_optimizer
from timm.models import create_model
from timm.scheduler.cosine_lr import CosineLRScheduler

from functools import partial
from tqdm import tqdm
from time import time

import matplotlib.pyplot as plt
from PIL import Image

import japanize_matplotlib
import models_mae

## MAEによる再構成画像の可視化

学習済みのEncoderとDecoderに対してマスクした画像を入力し，どのような再構成画像が作成されるのか確認します．

### 学習済みモデルのダウンロードとMAEモデルの定義

ImageNet-1Kを用いてMAEにより学習したEncoderとDecoderの重みパラメータをダウンロードします．\
その後，EncoderとDecoderを定義して，ダウンロードした学習済みの重みパラメータに上書きします．\
ここでは，EncoderとしてViT-Largeモデルを使用します．

In [ ]:
# 重みパラメータのダウンロード
!wget -nc https://dl.fbaipublicfiles.com/mae/visualize/mae_visualize_vit_large.pth

# EncoderとDecoderを用意（乱数による初期値）
model = getattr(models_mae, 'mae_vit_large_patch16')()

# 重みパラメータの読み込み
checkpoint = torch.load('mae_visualize_vit_large.pth', map_location='cpu')
msg = model.load_state_dict(checkpoint['model'], strict=False)
print(msg)

### 画像を表示する関数の定義

In [ ]:
def show_image(image, title=''):
    # image : [H, W, 3]
    assert image.shape[2] == 3
    plt.imshow(torch.clip((image * imagenet_std + imagenet_mean) * 255, 0, 255).int())
    plt.title(title, fontsize=16)
    plt.axis('off')
    return

### 入力画像の用意
入力画像をImageNet-1Kのvalidationデータから１サンプルをダウンロードすることで用意します．\
img_urlを変えることでダウンロードする画像を変更することでができ，ここではアヒル，ウサギ，イチゴのクラスの画像がダウンロード可能なURLを用意しています．

In [ ]:
# 画像がダウンロード可能なURL
img_url = 'https://dl.dropboxusercontent.com/s/bcb12avyq2989nt/duck.jpg' # class：アヒル　（n07745940）
#img_url = 'https://dl.dropboxusercontent.com/s/j04yg1b1jvqbkh8/rabbit.jpg' # class：ウサギ　（n02326432）
#img_url = 'https://dl.dropboxusercontent.com/s/m00c2s9cquawi1b/strawberry.jpg' # class：イチゴ　（n01847000）

# 画像のダウンロード (requests.get) と読み込み (Image.open)
img = Image.open(requests.get(img_url, stream=True).raw)

# 画像のリサイズ
img = img.resize((224, 224))

# 画像の正規化
img = np.array(img) / 255.

# ImageNet内の画像における色情報の平均値と標準値
imagenet_mean = np.array([0.485, 0.456, 0.406])
imagenet_std = np.array([0.229, 0.224, 0.225])

# 平均値と標準偏差を用いて正規化
img = img - imagenet_mean
img = img / imagenet_std

# 入力画像の表示
plt.rcParams['figure.figsize'] = [3, 3]
show_image(torch.tensor(img))


### マスクの設定
パッチ単位のマスク処理は，パッチごとにマスクの適用の有無を判定します．\
今回は，マスク率を0.75とし，75%の確率でマスク処理が適用されるように設定します．

In [ ]:
# マスク率の指定
mask_raito = 0.75

# seed値の設定
torch.manual_seed(2)

### 再構成画像結果の表示
用意した画像をEncoder・Decoderへ入力します．\
Encoder・Decoderのプログラムは，特徴量の抽出とピクセルの予測だけではなく，マスク処理，損失計算を内包する形で定義されています．\
そのため，Encoder・Decoderへ入力画像とマスク率を入力することで，パッチ分割，マスク処理，マスクを適用した画像に対する特徴抽出，特徴量からピクセルの予測，損失計算が順番に実行されます．\
最終的にEncoder・Decoderは，損失値（マスクしたパッチに対する再構成結果と実際のパッチ間のMSE），パッチごとの再構成結果，パッチごとのマスクを出力します．

In [ ]:
# 画像をPytorchで扱う場合の形式へ変更
x = torch.tensor(img)  # Tensor形式へ変更
x = x.unsqueeze(dim=0)  # データ数に関する軸の追加
x = torch.einsum('nhwc->nchw', x)  # 軸の順番の変更 [データ数,縦,横,RGB]->[データ数,RGB,縦,横]

# 画像をEncoder・Decoderへ入力
loss, y, mask = model(x.float(), mask_raito)  # loss：損失値，y：再構成画像，mask：マスク

# マスク適用の有無を表すTensorからマスクを作成 [データ数,パッチ数]->[データ数,パッチ数,パッチの画像サイズ(H×W×C)]
mask = mask.detach()
mask = mask.unsqueeze(-1).repeat(1, 1, model.patch_embed.patch_size[0]**2*3)  # パッチサイズに合わせたマスクを作成

パッチごとの再構成結果とパッチごとのマスクをmatplotlibを用いて可視化します．\
MAEでは，全てのパッチで損失計算は行わず，マスクしたパッチに対してのみ損失計算を行います．\
そのため，マスクしていないパッチの再構成結果は，マスクしたパッチの再構成結果と比べて品質が悪いという傾向にあります．\
そこで，MAEの論文内では，マスクした入力画像のマスク部分へ再構成したパッチを当てはめる可視化をおこなっています．

今回は，原画像（入力画像），マスクを適用した入力画像，再構成画像，マスクされたパッチへ再構成したパッチを当てはめた画像の4つを可視化を行い，学習によりEncoder・Decoderがマスクされていないパッチの情報からどの程度マスクされたパッチを予測できるようになっているのかを確認します．

In [ ]:
# 入力画像をmatplotlibで表示可能な形式へ変更
x = torch.einsum('nchw->nhwc', x)

# 再構成画像をmatplotlibで表示可能な形式へ変更
y = model.unpatchify(y)  # パッチごとの画像を1つの画像へ結合
y = torch.einsum('nchw->nhwc', y).detach().cpu()  # 軸の順番の変更 [データ数,RGB,縦,横]->[データ数,縦,横,RGB]

# マスクをmatplotlibで表示可能な形式へ変更
mask = model.unpatchify(mask)  # パッチごとのマスクを1つのマスクへ結合
mask = torch.einsum('nchw->nhwc', mask).detach().cpu()  # 軸の順番の変更 [データ数,RGB,縦,横]->[データ数,縦,横,RGB]

# マスクを適用した入力画像の作成
im_masked = x * (1 - mask)

# マスクされたパッチへ再構成したパッチを当てはめた画像の作成
im_paste = x * (1 - mask) + y * mask

# 再構成画像の表示 （配置やサイズ感などの設定）
plt.rcParams['figure.figsize'] = [24, 24]
plt.subplot(1, 4, 1)
show_image(x[0], "原画像")
plt.subplot(1, 4, 2)
show_image(im_masked[0], "マスクを適用した入力画像")
plt.subplot(1, 4, 3)
show_image(y[0], "再構成画像")
plt.subplot(1, 4, 4)
show_image(im_paste[0], "マスクされたパッチへ\n再構成したパッチを当てはめた画像")
plt.show()

## ImageNetで事前学習したモデルのfine-tuning

教師あり学習による事前学習モデルと自己教師あり学習 (MAE) による事前学習モデルをfine-tuningした場合の性能を比較することで，事前学習の方法による違いを評価します．\
今回は，ネットワークとしてViT-Baseモデル，事前学習としてImageNetデータセット，fine-tuningとしてCIFAR-10データセットを使用します．\
しかし，ImageNetデータセットを用いた学習は多くの時間を必要とします．\
そこで，教師あり事前学習モデルは様々な画像認識モデルの実装と学習済みモデルが利用可能なライブラリtimmで公開されている重みパラメータ，MAEによる自己教師あり事前学習モデルはMAEの著者らが公開している重みパラメータをダウンロードして利用します．

### データセットの準備
fine-tuning先のデータセットとして，CIFAR-10データセットを用意します．\
事前学習では画像サイズを224×224として学習を行なっています．\
そのため，位置埋め込みで埋め込まれる位置情報は，224×224の画像サイズにおけるパッチ数に合わせたサイズとなっています．\
このような事前学習時とfine-tuning時に画像サイズが異なる場合は，「位置埋め込みのサイズをfine-tuning時の画像サイズにおけるパッチ数に合わせてResizeする方法」と「fine-tuning時の画像サイズを事前学習時の画像サイズへResizeする方法」のどちらかを行います．\
今回は，fine-tuning時の画像サイズを事前学習時の画像サイズへリサイズする方法を適用します．\
CIFAR-10データセットの画像サイズは32×32のため，fine-tuning時は224×224にResizeして利用します．


In [ ]:
# 学習用データのデータ増幅と評価用データのデータ増幅の設定
train_transform = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                      transforms.Resize(224),
                                      transforms.RandomHorizontalFlip(),
                                      transforms.ToTensor(),
                                      transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
                                     ])
test_transform  = transforms.Compose([transforms.Resize(224),
                                      transforms.ToTensor(),
                                      transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
                                     ])

# 学習用データと評価用データの用意
dataset_train = torchvision.datasets.CIFAR10("./", train=True, transform=train_transform, download=True)
dataset_test  = torchvision.datasets.CIFAR10("./", train=False, transform=test_transform, download=False)

# 学習用データのDataloaderと評価用データのDataloaderの用意
dataloader_train = torch.utils.data.DataLoader(dataset_train, batch_size=70, num_workers=8, pin_memory=True, drop_last=True)
dataloader_test  = torch.utils.data.DataLoader(dataset_test, batch_size=70, num_workers=8, pin_memory=True, drop_last=False)

### 教師あり事前学習モデルのfine-tuning
timmライブラリのcreate_model関数を使用して教師あり事前学習済みのViT-Baseモデルを用意します．

In [ ]:
ViT_finetune = create_model('vit_deit_base_patch16_224', pretrained=True, num_classes=10)  # pretrained=Trueとすることで教師あり学習済みのモデルを用意

#### 学習条件の設定

In [ ]:
# エポック数の設定
epochs = 10

# 学習率の設定
lr  = 0.0001

# 最適化方法の設定
weight_decay = 0.05
optimizer_ViT     = torch.optim.AdamW(ViT_finetune.parameters(), lr=lr, weight_decay=weight_decay)

# 学習率の減衰方法の設定
warmup_t = 0
lr_scheduler_ViT  = CosineLRScheduler(optimizer=optimizer_ViT, t_initial=epochs, warmup_t=warmup_t)

# 損失式の設定
criterion = torch.nn.CrossEntropyLoss()

#### ネットワークのfine-tuningと評価

In [ ]:
# ネットワークをGPUへ
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ViT_finetune.to(device)

# 学習時間短縮のためにAutomatic Mixed Precision(amp)を利用
use_amp = True
scaler_ViT = torch.amp.GradScaler('cuda', enabled=use_amp)

start = time()
for epoch in range(epochs):
    # ネットワークを学習モードへ変更
    ViT_finetune.train()

    # ログ用の設定
    sum_loss = 0.0
    count    = 0

    for img, cls in dataloader_train:
        # 学習用データをGPUへ
        img = img.to(device, non_blocking=True)
        cls = cls.to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            # ネットワークに学習用データを入力
            logit_ViT = ViT_finetune(img)
            # 損失を計算
            loss_ViT  = criterion(logit_ViT, cls)

        # ネットワークの更新
        optimizer_ViT.zero_grad()
        scaler_ViT.scale(loss_ViT).backward()
        scaler_ViT.step(optimizer_ViT)
        scaler_ViT.update()

        # ログ用に損失値と正解したデータ数を取得
        sum_loss += loss_ViT.item()
        count    += torch.sum(logit_ViT.argmax(dim=1) == cls).item()

    # 学習率の調整
    lr_scheduler_ViT.step(epoch)

    # ログの表示
    print(f"epoch: {epoch+1},\
            mean loss: {round(sum_loss/len(dataloader_train), 3)},\
            mean accuracy: {round(count/len(dataloader_train.dataset), 2)},\
            elapsed_time : {round(time()-start, 2)}")

    # ネットワークを評価モードへ変更
    ViT_finetune.eval()

    # ログ用の設定
    count = 0

    # ネットワークの評価
    with torch.no_grad():
        for img, cls in dataloader_test:
            # 評価用データをGPUへ
            img = img.to(device, non_blocking=True)
            cls = cls.to(device, non_blocking=True)

            # ネットワークに評価用データを入力
            logit_ViT = ViT_finetune(img)

            # 正解したデータ数をカウント
            count += torch.sum(logit_ViT.argmax(dim=1) == cls).item()

        # 評価用データに対する正解率を計算して表示
        print(f"test accuracy: {count/len(dataloader_test.dataset)}")


### MAEによる自己教師あり事前学習モデルのfine-tuning
timmライブラリのcreate_model関数を使用してViT-Baseモデルを用意し，MAEにより学習した重みパラメータを読み込みます．

In [ ]:
# 重みパラメータのダウンロード
!wget -nc https://dl.fbaipublicfiles.com/mae/pretrain/mae_pretrain_vit_base.pth

# ViT-Baseモデルの用意
MAE_finetune =create_model('vit_base_patch16_224', pretrained=False, num_classes=10)

# 重みパラメータの読み込み
checkpoint = torch.load('mae_pretrain_vit_base.pth', map_location='cpu')
msg = MAE_finetune.load_state_dict(checkpoint['model'], strict=False)
print(msg)

### 学習条件の設定

In [ ]:
# エポック数の設定
epochs = 10

# 学習率の設定
lr  = 0.0001

# 最適化方法の設定
weight_decay = 0.05
optimizer_MAE     = torch.optim.AdamW(MAE_finetune.parameters(), lr=lr, weight_decay=weight_decay)

# 学習率の減衰方法の設定
warmup_t = 0
lr_scheduler_MAE  = CosineLRScheduler(optimizer=optimizer_MAE, t_initial=epochs, warmup_t=warmup_t)

# 損失式の設定
criterion = torch.nn.CrossEntropyLoss()

#### ネットワークのfine-tuningと評価

In [ ]:
# ネットワークをGPUへ
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAE_finetune.to(device)

# 学習時間短縮のためにAutomatic Mixed Precision(amp)を利用
use_amp = True
scaler_MAE = torch.amp.GradScaler('cuda', enabled=use_amp)

start = time()
for epoch in range(epochs):
    # ネットワークを学習モードへ変更
    MAE_finetune.train()

    # ログ用の設定
    sum_loss = 0.0
    count    = 0

    for img, cls in dataloader_train:
        # 学習用データをGPUへ
        img = img.to(device, non_blocking=True)
        cls = cls.to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            # ネットワークに学習用データを入力
            logit_MAE = MAE_finetune(img)
            # 損失を計算
            loss_MAE  = criterion(logit_MAE, cls)

        # ネットワークの更新
        optimizer_MAE.zero_grad()
        scaler_MAE.scale(loss_MAE).backward()
        scaler_MAE.step(optimizer_MAE)
        scaler_MAE.update()

        # ログ用に損失値と正解したデータ数を取得
        sum_loss += loss_MAE.item()
        count    += torch.sum(logit_MAE.argmax(dim=1) == cls).item()

    # 学習率の調整
    lr_scheduler_MAE.step(epoch)

    # ログの表示
    print(f"epoch: {epoch+1},\
            mean loss: {round(sum_loss/len(dataloader_train), 3)},\
            mean accuracy: {round(count/len(dataloader_train.dataset), 2)},\
            elapsed_time : {round(time()-start, 2)}")

    # ネットワークを評価用モードへ変更
    MAE_finetune.eval()

    # ログ用の設定
    count = 0

    # ネットワークの評価
    with torch.no_grad():
        for img, cls in dataloader_test:
            # 評価用データをGPUへ
            img = img.to(device, non_blocking=True)
            cls = cls.to(device, non_blocking=True)

            # ネットワークに評価用データを入力
            logit_MAE = MAE_finetune(img)

            # 正解したデータ数をカウント
            count += torch.sum(logit_MAE.argmax(dim=1) == cls).item()

        # 評価用データに対する正解率を計算して表示
        print(f"test accuracy: {count/len(dataloader_test.dataset)}")


## 課題
1.   入力画像を変更してみましょう
2.   マスク率を変更してみましょう